# Collect Qwen experiment logs for the final write-up

**CPU Colab; no GPU, API token, model download, or package installation needed.**
Run all cells. Mount Drive, collect the existing files, build and verify one ZIP,
save a copy on Drive, and trigger a browser download. Send that ZIP for analysis.

Includes all seven trained variants, the distillation extension (epochs 6–7),
and the original instruct-RL history needed for steps 1–40 before recovery.
Loss, reward, sequence length, entropy, gradient norm, learning rate and timing
fields are retained exactly as recorded, including nested metrics. Available
per-epoch math evaluation tables, run manifests and checkpoint JSON metadata
are included for interpreting the training curves.

**No model weights, optimizer states, datasets, raw rollouts, generated answers,
console logs or credentials.** Sources on Drive are read only. No graphs or
merged histories are generated yet. Missing files are reported rather than
silently treated as complete. RunPod/HF transfer results are separate artifacts.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
LOCAL_OUTPUT = Path("/content/qwen-training-log-export")
MAX_FILE_MB = 100  # Larger individual files are skipped and listed in the report.
SAVE_COPY_TO_DRIVE = True
DRIVE_EXPORT_ROOT = DRIVE_ROOT / "LG-AIME-Qwen25-3B-Experiments/writeup-artifacts"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import base64, io, sys, zipfile

CODE_ROOT = Path("/content/qwen-log-collector")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVzLEEfq/g4AAOUpAAAlAAAAc2NyaXB0cy9jb2xsZWN0X3F3ZW5fdHJhaW5pbmdfbG9ncy5web1afW/bxhn/n5/ikCEglUm0k6bt4EwD3MZpvcVO5qTd1iAgKPEoXU3yNB5pWzEM7DvsG+6T7Pc8d0dSL45TYJiAxOLx7nl/Pz169OhCptlEV8VazHVRyHmjdCV0Lv56LSshb1ayVqWsGlHohRG5rkWzlCJXVVqIVa1/xQFxXatGTtpVHATvmrTK0joThZrVab0WBPmFkOVMZpnMhKr4/PdvfxLf6yKdDZFWupEzrS/j4NGjR0GgypWuGzE3V/5rXpVpM1/6x2VqlkDjH5X23341uvLftfHfPqlVrgoZ5LUuRZY2sgFjwr30z2NB/3/Sldu3ShvC4be9xWMQnPz97cnF6dnJ+ft3YirC1z9MjvE0IYk9+3ry1XeTk05sJgwufjqnbR8CgU8UzlIjk7oIx4K/TupiUqWNupITqQ2tZulKJ/9kWMlXs2RVy6ZOVSWzhPbTDpwejR24S1WqxORNBw/fJ1fPJmYJemkRzwNojN08TeiYe9vDquWvVhUbALvViUnLVaGqxcS93gdaFnlSd6972JkyjSqKDcjfziZuObXw7wX77SxxOz8HOJErPV+ab5JvP4diwru+njQa2xjIAyiTua4aVbXyMwIbKnSvvCZ2x5ZyewBM1HMvnqGGPRUDFPu42gt/wITleh8CVZmmbueNwxDHB96iXx6/fTMw6zNVqe/IA786PJycyVLX61fqZg/afYY6wJLoWi0ognw5ugeRfAzeXxyfnidvj9+/P7kYeFxoGrkyMcUExle3VVKmlcqlaXjVgm5SKIN8vFurpdHFFdwO6s/Vwq1bmG5pnZYMkx0Uik5MW5aIeh0MfpGU8GA1N91q2jSyXDUdURamW02e9PtmCKaaPJ95cG8YI2szeZLsQwBZnPx8/Do5Pz47ITncWvj2yDaFm6sItmzk9jFp0lkhu62bq7zVS6NcpbXCtq0TOy8c/H59z9YhZNPOyEFw2lwms1qml5m+royHsy1Y/2wh3AXnb96zBJBPfmcTmtcUpbNB7gmCi/RaUHowIq0lMps0sobyRVtlshZZjQB9IK5Vs6T8pWrhTZgzhBG1LDiIi0aLs/VL2h4HPfxNi0NyNI0RRrf1XFoAY0Yu5ks5v4Sc8VwqY4hQJmosZF3rGsRVWWDUJ4nMPC9aA9AmFmTSqrpCutFWhw5BepWqguQuyH7AbouEdFCn1UIyJGHlBRSyyEwcnGsxb8G1QWoFKzMJgZlSa3CcgWFsrxcyG9ujjjwncpYaEjgEAd+wni4zFASnFehE5j8SRl55BQDcFZSdgpyxGEYy4UKtfbDhXHzzn3/9+9sx1w1e6oGPJpOL14zySqLWgAFBBuNeyUY2gLYAz+zaQMYCXmmFgubP796cE/kpOf84IKZ6gaHQWAp5lRaOFGv7CqKJFHNE4JHkbTIBCaYtGjOKg+8OWTynh1aGle6pWYJRTTBeOBuCzc9YGNvY2FmsTJWTX3C9hPTYLqsmFhdt9Va7ug0IKpPDSh0VfA6mj9pLsCUyRV5BuUREBc6lrOWLoDWSBZsWcK5sPUHBB0Hh9Y/tYkFEv0phohYwQDQqT+dkuhKyRskE7LYk1IawpMXaSMNat+oRF6+PBOHo3IUjmXgKlT4/ZJ1Wossk7uXzp6D8GkVkLN5sHAvSvAGbzw/FNYgXXXiMxV8k7Jv4aK41uUNPIRwY1jWwkRp/q7QCV8pYcyA7fTkwwqOeWmeBRO7XLMV77dOqu8MKGoT3efMiyDTLPjWwI+m03/ukhGNDioVMazKUCUMwpKWWzADSbyD6A1+GCJXBBtQ8LYp1HBx7KUwYYB/DMpUuKg225s7y1mNLRJYpIp+k2s7n0pi8LUS7ohIYuiOHQtzSbQMKwScBgKDSCnSsZIrz81obELxCpYxIVPujwizhITCZ1ANg1QfuIclVbZrEbj6Yp8Re4jEhaqZXWmVWHgSElIn6SYqFrCQJJBNcTgFPc63miK7ByY0PLaXOZAGjUIslOXlH2oEXYn1wcf4D80OxtNGXsmLSXXClCIBYEZgqXaFwJhg18oGjDk89EQMvhdfBDvHWyDmKdDP2FoK6QXK/FAdvyEFNCVUNQw9CySVosrGwTpguG4969ZE/Cu5BBgdzXSAdAa71a5tBXPBFeK1JEgrhHC7xiiEhTUiZdQKVlI9IlJRCCgipcbis18fBaU7qg/HCOVyIJy/F0y+nb2kxpSjbKBhPn92sXcGz0DY2S5gIwFpyEJlcNxdkMhfXaXGZMIsRNjejI87yeH/Myh/IHEFNuiRl1iWK6EsjrpeUIg1MsdphKFPwcBtdY8JHcFVuMxIwxcqgEK6jEfmZX3CAI0cGfaDGtq74keKahwrPcZIfOw2hj9UmJn6YE35f6GsmdPoqLYwcQHVnPxx9RC1iWNJRzghyAuReE72WYqzdht74qK5JktWaPSZJuGTm7+Fdh2Dfh0RHwCLqWaOOk5E4EPlog/3RgNBaVOR0oMDTSewOeKEPVSzgZBcwnd3Y6VRAB+6RuP+sqQbhjc5WKJr5ujSCZbgztlYZ24Qw5rhe0zNcDNSXMIUppf1oNBYfPo4FcsxhwAcJ5pUyqonIg9HoI5fm6mYahhuaqsWlhLoVanGSAu+N6clsU82Cmoo8vLWQ7uJbHL0LiWm7IiTsgOBtywRkIEFSEooINNVA82aPUCy9dguhG23sIOi7Z1wxhygf2SOdNUP0rFkIM84gt0xGYdvkkz+Eo9igUW1owwabTn20HiOhq9W2DHxK6haber25wyoIcqLCNy50mpmI4I322clALvbcvZJB5IRofybtnFBlHIU0c+FQoLllCHsEFANXzWDzJrjecn4/FU8/z54zt82dVkv21Wgoux124gUskxtStKxUSVLggEya0RaLtmdNVytZZe7sB3vuo8VgAxWFCUtReNS7Am9MSlVhEf9HDGxEBNn6iq3yHCXDeAOnO5be0LH05jPH0C+3sJc5shZ3pklPRCE7fBP7AI7twmgLXSf3hO2OsbqVrpVLrDXjXReO6Hl054KEy0ER92eJjcQImiuUGwhMY/GEAsMNp5yknE2fHh5yrlL5erpCreItqz/ehbVuZRS7QUA0GmaVfkOXWwZphK2TMvC5bl6hpMmskebhGdU34getF0hlNolzXXQkbnuId2GHakC9+ONUHG7jGHrAcG/ZmgYNnFhp2CbgOoi9bDyj/cr2jri8JLZsU2Wm72sKmvIGBWGiL/nRnoCFlyuA8yPUuNLXkZ+ixm0zH1HsyGklCh//43H5OHv/+MfHZ4/fPc5/cXRlKHFVZcuq6ZBKJKuQRj5JN2Khuiq5ZaR38Se1CgPrxLbWpnFHiGqMarUEuMlyaCvZbCdfXqyHOsZrTnR4Q1mDhkTu+07kCV1Z5LeaSwU3zfyj7dP900Al1qv8k83dhjpsl694wZZkU0TpJhqq84l4evjs+ZMnz+w2nkO4eXb8i1qRpUUDGQIzjeW6Hadvk5cnr14fvz95Oaauk9pIU6BCK6bfjNCVoAicLyGM3rrIuSiFUEJmN4JEptb52Zt3ahxnr1waUPUAzvZE7b686iIdNsYe02ZS2MklA/fjiqJzTHJBP4FJGj3ULBV8X1B+DG3oQ6fTjz4E34YEw9kNU8oDwpTmTkcipDoNTRlXloTQIQrvRnsR7REDi4KK8qmlllqCiBwnodV9UuDdf7IW87/mp28PeOS8btgzaPW3cfQ78R0FP+5l0LdpPSbbmKc0LOAJ3KLW176wp3reXwFxb+76C5T6qol3YLMPsKw0WIrCehayKYMjmZb7JUINxtTtiImoyDrc78XTXb4gY0pgVHr+H+T82wTrak8b0w5CMOAcYq8PxKlJKBHcRLtInOfHfIdHVBJobn939/aM2mA5YNN2oluMOtiJk4GF3JmTF+5uiL3/E5pl+uzrb3DaXf7FdoEBxUt5k6kFCIxGe8TpC5l7TKOtYjubiHY6j3tU01a2JjOJM1XSPLRCqXGzULcFaPTmHWfqsfipUlR+u6c+h7MBy90SdVP4LsV83sh4k1vj7ySSjfBOk4ykm2sOG3L/yV3sJKfd37pvyRe1bakvZUgHdgzSmiJND3jaFS7zxE4bTb+f394TnneK8SFiPu1b18HtBxDZd6bNqSObwmnsdQE3yB4pzVStT0354ofGqOFvsEv+EMAwb4viQf63y/0OQpcHdztSOyUYOy+6VBVfptPl8lbDZavYgf+jiBreWB84UNuFrf/wRf9UuJNRSM8HyEcL+ZR7W0ZNcqKrRNsV8J5w7wSAZklUk4EHWCNzEhIEesAfCod9ReZrsW2vOhJcbmwOPHqH4Gqtcwc8jXbMWPruEDL7wBwebN8M+tXdC8KP95YhHixVF+QcD5cWvnrsyGUHdWBGu4Tv8T8idL//pdU6cj+SiN1fSra9d4wJXiPrauSB0wPB37w9fcA+bTmIf74atBOhqdgn2S/iyVlbqKoVzbycJe3ncsPXb7vL1+429gtvd/vHu4e8cbjaT2M3vGSw/JCzDLbu95l7ZDQ49+WC8ePl/haXiUt6YAMxrfTO+m+SzZ6UQkql5S0zsE04GmKemaEUQgkT5Y9uKQ3zkMSPFMbicHQ0vmM7m1DJ6EcwXXR49GCchmjuSdZWIWen796dnv8AUtuKqsL+yhQIX1CL4ibk4WgrMttIykGFuZzwzzO6y8m6mKTlfELXexO63OefMfA++4sNvrKhH1Ns7doKOFtifTiyd6cHpSmMoDVcn0ShG+GHdsKwE5r4rnen3Nhc7UrdB6SPQxSVaqvV7ZkPaZcFyTLcjugj7zCaLn4aObChnbI1vDg5fnl2EpcZgPK9/+c233czj7M8p8zacmUi/4q6Frpymz4bBGi+o6VWXcfvGrr3On0zSKRA0JYVRYgPnPp81vP5bk+S46Dkjb6f4Y0Hg7m9pfJ9w7jx7ohtZ67Wa49lQ1OhubmKX6p58zdeiJjNsZ0qU2gxU8faaOuole4S7oNTm1cKfJ+0q95NIx9CQVcY3T55wglmZxI4UFAXLTb30Oh/NGwBdg1g5ycTOGR/ogJ4PPT39yIPzVr2z09mKc28PV5C+omm58PpMLZsVW7bE7085Eu3qpELEL62qUfkKSjIjsQtAPhJoRsHb4yAvLiD/wJQSwECFAMUAAAACAAAACFcyxBH6v4OAADlKQAAJQAAAAAAAAAAAAAAgAEAAAAAc2NyaXB0cy9jb2xsZWN0X3F3ZW5fdHJhaW5pbmdfbG9ncy5weVBLBQYAAAAAAQABAFMAAABBDwAAAAA='
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as archive:
    archive.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT / "scripts"))
from collect_qwen_training_logs import collect

In [ ]:
import shutil

ZIP_PATH, report = collect(DRIVE_ROOT, LOCAL_OUTPUT, max_file_mb=MAX_FILE_MB)
print("\nCollection:", report["status"])
print("Files:", len(report["files"]))
print("ZIP:", ZIP_PATH, f"({ZIP_PATH.stat().st_size / 1024**2:.1f} MiB)")
for category in ["missing", "skipped", "errors"]:
    if report[category]:
        print("\n" + category.upper() + ":")
        for item in report[category]:
            print(item)
for run in report["runs"]:
    if run.get("malformed_lines"):
        print("WARNING:", run["key"], "has malformed step-log lines; raw bytes are preserved.")
if SAVE_COPY_TO_DRIVE:
    DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
    drive_copy = DRIVE_EXPORT_ROOT / ZIP_PATH.name
    shutil.copy2(ZIP_PATH, drive_copy)
    print("\nDrive backup:", drive_copy)

## Download and send the ZIP

The ZIP contains `README.md`, `collection_manifest.json` (file hashes and missing
paths), `run_inventory.csv` (available metrics and step ranges), and the original
files under `drive/`. A partial collection can still be analyzed; missing paths
help identify what needs to be located later.

The next cell starts the browser download. If it is blocked, rerun only that
cell or download the ZIP from Colab's Files sidebar or the printed Drive backup.

In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))